# DATA Hypothesis - VANGUARD A/B TEST
---

** Import Libraries**

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

sns.set_style("whitegrid")

**Loading Database**

In [2]:
data_path = "data/processed"
web = pd.read_csv(f"{data_path}/web.csv")

## Phase 2 KPIs 

**GLOBAL COMPLETION RATE**
- Number of clients who finished the process in correct order, meaning the client reached the confirm step without errors: clients who went through the whole process smoothly

**COMPLETION RATE**
- Number of correct: if user managed to go through the process with less repetitions and errors, but still reached the confirmation step

**ERROR RATES**
- If there a step where users go back to a previous step. Moving from a later step to an earlier one is considered an error.

**ERROR RATE BY REPETITION**
- If there is a step where users repeatedly go back to. Moving back and forth on the same step. This metric serves as an indicator of possible UI errors in specific steps

**DROP-OUT RATE**
- Number of clients that didn't manage to reach the last stage (confirm) and abandoned the process before

**TIME SPENT ON EACH STEP**
- The average duration users spend on each step.

GLOBAL COMPLETION RATE

In [3]:
# 1. Define the ideal sequence
ideal_path = ['start', 'step_1', 'step_2', 'step_3', 'confirm']

# 2. Logic to check each group
def check_journey(group):
    actual_steps = group['process_step'].tolist()
    actual_statuses = group['process_status'].tolist()
    
    # Check if sequence is exactly the ideal path AND all statuses are 'correct'
    return (actual_steps == ideal_path) and all(s == 'correct' for s in actual_statuses)

# 3. Calculate per client (This creates a Series with client_id as index)
global_success_map = web.groupby('client_id').apply(check_journey)

# 4. Map the result back to every row in the original dataframe
web['global_success'] = web['client_id'].map(global_success_map)

print(global_success_map)

client_id
555         True
647         True
934        False
1028       False
1104       False
           ...  
9999150    False
9999400     True
9999626    False
9999729    False
9999832    False
Length: 50500, dtype: bool


C:\Users\eymem\AppData\Local\Temp\ipykernel_6908\1463843925.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  global_success_map = web.groupby('client_id').apply(check_journey)


COMPLETION RATE (stored under variable is_completed)

In [4]:
# 1. Define the success logic for Completion Rate
def check_any_completion(group):
    """
    Checks if 'confirm' exists in the user's journey.
    """
    actual_steps = group['process_step'].tolist()
    
    # Standard Completion: Did they ever reach the end?
    return 'confirm' in actual_steps

completion_map = web.groupby('client_id').agg(
    is_completed=('process_step', lambda x: 'confirm' in x.tolist()))

# Map back to original dataframe
web['is_completed'] = web['client_id'].map(completion_map['is_completed'])

# Calculate completion rate
completion_rate_total = web.drop_duplicates('client_id')['is_completed'].mean() * 100
print(f"Overall Completion Rate: {completion_rate_total:.2f}%")

Overall Completion Rate: 67.57%


VARIATION OF COMPLETION RATE (INCL. POSSIBLE ERRORS)

In [5]:
# Step 1: Calculate completion per client per variation
completion_by_user = web.groupby(['variation', 'client_id']).agg(
    is_completed=('process_step', lambda x: 'confirm' in x.tolist())
).reset_index()

# Step 2: Calculate mean per variation
completion_rate_report = (
    completion_by_user.groupby('variation')['is_completed']
    .mean() * 100
)

# Step 3: Print nicely
print("--- Overall Completion Rate by Variation ---")
print(completion_rate_report.round(2).apply(lambda x: f"{x}%"))

--- Overall Completion Rate by Variation ---
variation
Control    65.59%
Test       69.29%
Name: is_completed, dtype: object


ERROR RATES (BACKWARDS ERROR) & (REPETITON ERROR)

In [6]:
web["backwards_error"] = web["process_status"].map({
    "correct": False,
    "repeated": False,
    "error": True}).astype("Int64")  # now True→1, False→0

In [11]:
error_rate_report = (
    web.groupby(['variation', 'process_step'])['backwards_error']
    .mean() * 100
)

print("--- Error Rate by Variation ---")
print(error_rate_report.round(2).apply(lambda x: f"{x}%"))

--- Error Rate by Variation ---
variation  process_step
Control    confirm          4.06%
           start           20.78%
           step_1          20.37%
           step_2           14.2%
           step_3          28.44%
Test       confirm          1.21%
           start           18.01%
           step_1           24.8%
           step_2          19.82%
           step_3          25.24%
Name: backwards_error, dtype: object


In [ ]:
web.head()

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,sec_spent,variation,global_success,is_completed,backwards_error
0,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,7.0,Test,True,True,0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,32.0,Test,True,True,0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,99.0,Test,True,True,0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,20.0,Test,True,True,0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaN,NaN,Test,True,True,0


OVERALL DROP-OUT RATE

In [ ]:
# 1. Identify the last step for each client
last_steps = web.sort_values(['client_id', 'date_time']).groupby('client_id')['process_step'].last()

# 2. A drop-out is anyone whose last step is NOT 'confirm'
is_dropout = last_steps != 'confirm'

# 3. Calculate Rate per variation
dropout_report = is_dropout.to_frame().join(web[['client_id', 'variation']].drop_duplicates().set_index('client_id'))
dropout_rate = dropout_report.groupby('variation')['process_step'].mean() * 100

print("--- Drop-Out Rate (%) ---")
print(dropout_rate.round(2))

--- Drop-Out Rate (%) ---
variation
Control    41.87
Test       32.68
Name: process_step, dtype: float64


DROP-OUT RATE BY STEP

In [ ]:
# 1. Sort globally to ensure the last row is the chronologically final interaction
web = web.sort_values(['client_id', 'date_time'])

# 2. Get the total unique clients per variation (Denominator)
total_clients_per_group = web.groupby('variation')['client_id'].nunique()

# 3. Identify the final step for every client
# .tail(1) after groupby gives us the very last record for each client
final_steps_per_client = web.groupby('client_id').tail(1)

# 4. Count where users ended their journey
drop_off_counts = final_steps_per_client.groupby(['variation', 'process_step'])['client_id'].count().unstack(fill_value=0)

# 5. Calculate Drop-off Rate (%)
# This shows: "What % of total clients in this group stopped at this specific step?"
drop_off_rate_by_step = (drop_off_counts.div(total_clients_per_group, axis=0) * 100)

# 6. Reorder for logical funnel flow
step_order = ['start', 'step_1', 'step_2', 'step_3', 'confirm']
drop_off_rate_by_step = drop_off_rate_by_step[[col for col in step_order if col in drop_off_rate_by_step.columns]]

print("--- Drop-off Rate by Step (Final Interaction per Client) ---")
display(drop_off_rate_by_step.round(2))

--- Drop-off Rate by Step (Final Interaction per Client) ---


process_step,start,step_1,step_2,step_3,confirm
variation,,,,,
Control,23.72,9.46,3.12,5.58,58.13
Test,20.68,6.13,2.11,3.76,67.32


TIME SPENT ON EACH STEP

In [ ]:
# 1. Ensure chronological order within each session
web['date_time'] = pd.to_datetime(web['date_time'])
web = web.sort_values(['client_id', 'visit_id', 'date_time'])

# 2. Group by BOTH client and visit to isolate sessions
# We use .diff() and .shift(-1) within the session group
web['seconds_spent'] = web.groupby(['client_id', 'visit_id'])['date_time'].diff().shift(-1).dt.total_seconds()

# 4. Summarize Average Time per Step and Variation
time_summary = web.groupby(['variation', 'process_step'])['seconds_spent'].mean()

print("--- Average Seconds Spent per Step (Per Session) ---")
display(time_summary.round(2))

--- Average Seconds Spent per Step (Per Session) ---


variation  process_step
Control    confirm         168.73
           start            66.83
           step_1           50.47
           step_2           92.00
           step_3          137.14
Test       confirm         243.69
           start            61.47
           step_1           60.68
           step_2           88.85
           step_3          129.64
Name: seconds_spent, dtype: float64

In [ ]:
web.columns

Index(['client_id', 'visitor_id', 'visit_id', 'process_step', 'process_order',
       'process_status', 'date_time', 'next_time', 'sec_spent', 'variation',
       'global_success', 'is_completed', 'backwards_error', 'seconds_spent'],
      dtype='object')

In [ ]:
cols_to_convert = ['global_success', 'is_completed']  # your boolean columns
web[cols_to_convert] = web[cols_to_convert].astype(int)

In [ ]:
web.rename(columns= {'global_success': 'is_global_success', 'backwards_error':'has_backwards_error' }, inplace=True)
web = web.drop('sec_spent', axis=1)

In [ ]:
web.head()

,client_id,visitor_id,visit_id,process_step,process_order,process_status,date_time,next_time,variation,is_global_success,is_completed,has_backwards_error,seconds_spent
0,555,402506806_56087378777,637149525_38041617439_716659,start,step_1,correct,2017-04-15 12:57:56,2017-04-15 12:58:03,Test,1,1,0,7.0
1,555,402506806_56087378777,637149525_38041617439_716659,step_1,step_2,correct,2017-04-15 12:58:03,2017-04-15 12:58:35,Test,1,1,0,32.0
2,555,402506806_56087378777,637149525_38041617439_716659,step_2,step_3,correct,2017-04-15 12:58:35,2017-04-15 13:00:14,Test,1,1,0,99.0
3,555,402506806_56087378777,637149525_38041617439_716659,step_3,confirm,correct,2017-04-15 13:00:14,2017-04-15 13:00:34,Test,1,1,0,20.0
4,555,402506806_56087378777,637149525_38041617439_716659,confirm,NaN,correct,2017-04-15 13:00:34,NaN,Test,1,1,0,NaN


In [ ]:
import os

# Define the output directory
output_dir = "data/processed"
os.makedirs(output_dir, exist_ok=True)

# Exporting both the transactional (web) and summary (clients) datasets
# Assuming your enriched dataframe is named 'web' and the summary is 'df_clients'
web.to_csv(f"{output_dir}/vanguard_client_summary.csv", index=False)

print("Exported:")
for f in os.listdir(output_dir):
    path = os.path.join(output_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f} ({size:,} bytes)")

Exported:
  df_exde.csv (3,302,143 bytes)
  df_web_3.csv (81,420,206 bytes)
  vanguard_client_summary.csv (41,859,699 bytes)
  vanguard_hypo.csv (59,093,847 bytes)
  web.csv (39,957,052 bytes)
